# 🪐 StellarScan : Model Training Notebook

This notebook trains two simple, honest, explainable models:

1. **Classifier** — predicts if a signal is a CONFIRMED planet, a CANDIDATE, or a FALSE POSITIVE.
2. **Anomaly Detector** — flags signals that look statistically unusual/weird, so a human can take a closer look.

We use **real data** from NASA's Exoplanet Archive (the Kepler Objects of Interest table). No fake data, no shortcuts.

Every section below has a heading so you know exactly what part of the pipeline you are looking at. Run the cells **in order, top to bottom**.

## SECTION 1: Install and Import the Tools We Need

Before we do anything, we need to bring in the Python libraries (toolboxes) that will help us:
- `pandas` -> for handling tables of data (like Excel, but in code)
- `numpy` -> for doing math on numbers
- `sklearn` (scikit-learn) -> for building the machine learning models
- `joblib` -> for saving our trained models to a file
- `matplotlib` -> for drawing simple graphs so we can see what is happening

In [ ]:
# These libraries already come pre-installed in Google Colab.
# We are just importing (loading) them so we can use them below.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

print("All libraries loaded successfully.")

## SECTION 2: Download the Real Dataset

We are downloading the **Kepler Objects of Interest (KOI) Cumulative Table** directly from NASA's Exoplanet Archive.

This is real data collected by the Kepler Space Telescope. Each row is one star signal that scientists already looked at, and it is already labeled as one of:
- `CONFIRMED` -> it really is a planet
- `CANDIDATE` -> it might be a planet, not confirmed yet
- `FALSE POSITIVE` -> it looked like a planet signal, but it was not

In [ ]:
# This is the direct link NASA provides to download the full KOI table as a CSV file.
nasa_data_url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+cumulative&format=csv"

# We use pandas to read this CSV file straight from the internet into a table.
raw_data = pd.read_csv(nasa_data_url)

# Let's check how many rows and columns we got.
number_of_rows = raw_data.shape[0]
number_of_columns = raw_data.shape[1]

print("Data downloaded successfully.")
print("Number of rows (stars/signals):", number_of_rows)
print("Number of columns (measurements):", number_of_columns)

### 2.1 Backup plan (only run this if Section 2 above fails)

Sometimes the NASA server can be slow or temporarily down. If the cell above gives an error, download the CSV manually from:
https://exoplanetarchive.ipac.caltech.edu/cgi-bin/TblView/nph-tblView?app=ExoTbls&config=cumulative

Upload it to Colab (left sidebar -> Files -> Upload), name it `cumulative.csv`, and run the cell below instead.

In [ ]:
# ONLY run this cell if Section 2 failed and you manually uploaded the file.
# raw_data = pd.read_csv("cumulative.csv")
# print("Backup data loaded.")

## SECTION 3: Look at the Data First (Exploration)

Before touching or changing anything, we should always **look at the data first**. This helps us understand what we are working with.

In [ ]:
# .head() shows us the first 5 rows of the table, so we get a quick preview.
raw_data.head()

In [ ]:
# Let's see how many rows belong to each label (CONFIRMED, CANDIDATE, FALSE POSITIVE).
# This tells us if our data is balanced or not.
label_counts = raw_data["koi_disposition"].value_counts()
print(label_counts)

## SECTION 4: Data Cleaning

Real-world data is messy. Before we can train a model, we need to clean it. Cleaning means:
1. Keeping only the columns (measurements) that are actually useful for prediction.
2. Removing rows that have missing/empty values in important columns.
3. Making sure the label column only contains the 3 categories we care about.

In [ ]:
# Step 4.1: Choose the columns (features) we will actually use to make predictions.
# Each of these is a real, meaningful measurement about the signal:
#
# koi_period    -> how many days between each dip in brightness (orbital period)
# koi_duration  -> how many hours the dip lasts
# koi_depth     -> how much the brightness drops during the dip
# koi_prad      -> estimated size of the planet (in Earth-radius units)
# koi_teq       -> estimated temperature of the planet
# koi_insol     -> how much starlight/energy the planet receives
# koi_model_snr -> signal-to-noise ratio (how clear/strong the signal is)
# koi_steff     -> temperature of the host star
# koi_slogg     -> surface gravity of the host star
# koi_srad      -> radius of the host star

feature_columns = [
    "koi_period",
    "koi_duration",
    "koi_depth",
    "koi_prad",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_srad"
]

label_column = "koi_disposition"

# Step 4.2: Make a new, smaller table with only the columns we need.
columns_we_need = feature_columns + [label_column]
clean_data = raw_data[columns_we_need].copy()

print("Shape before cleaning:", clean_data.shape)

In [ ]:
# Step 4.3: Remove any row that has a missing (empty) value in ANY of our chosen columns.
# A model cannot learn from missing numbers, so we must either remove or fill them.
# Here, we choose the simple and safe beginner approach: remove the row.

clean_data = clean_data.dropna()

print("Shape after removing missing values:", clean_data.shape)

In [ ]:
# Step 4.4: Double-check that our label column only has the 3 categories we expect.

unique_labels = clean_data[label_column].unique()
print("Labels found in the data:", unique_labels)

## SECTION 5: Separate the Features (X) and the Label (y)

In machine learning:
- **X** = the input measurements the model will look at
- **y** = the correct answer we want the model to learn to predict

In [ ]:
# X contains all our chosen measurement columns.
X = clean_data[feature_columns]

# y contains only the label column (the correct answer).
y = clean_data[label_column]

print("X shape (rows, columns):", X.shape)
print("y shape (rows,):", y.shape)

## SECTION 6: Split Data into Training Set and Testing Set

We never test a model on the same data it learned from — that would be like giving a student the exam answers beforehand.

So we split our data:
- **80%** for training (the model learns from this)
- **20%** for testing (we check how well the model does on data it has never seen)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## SECTION 7: Scale the Features

Our measurement columns have very different ranges. For example, `koi_period` might be a small number like 4, while `koi_depth` might be a huge number like 5000.

If we do not fix this, the model might think the bigger numbers are automatically more important, which is wrong. **Scaling** puts every column on the same fair scale.

In [ ]:
# StandardScaler transforms each column so it has an average of 0 and a normal spread.
scaler = StandardScaler()

# We LEARN the scaling rules ONLY from the training data.
scaler.fit(X_train)

# Now we APPLY those same rules to both training and testing data.
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")

## SECTION 8: Train the Classification Model

This is the model that will predict: CONFIRMED, CANDIDATE, or FALSE POSITIVE.

We use a **Random Forest**. In simple words: it builds many small decision trees (like a big group of simple yes/no questionnaires), and each tree votes on the answer. The majority vote becomes the final prediction. This is beginner-friendly because you can literally look inside and see which questions it asked.

In [ ]:
# n_estimators = how many decision trees to build (more trees = usually more stable, but slower)
# random_state = keeps our results reproducible (same result every time we run it)

classifier_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

# .fit() is where the actual learning happens.
classifier_model.fit(X_train_scaled, y_train)

print("Classifier training complete.")

## SECTION 9: Check How Good the Classifier Is

Now we test the model on the 20% of data it has never seen before, and check how accurate it is.

In [ ]:
# Ask the model to make predictions on the test data.
predictions = classifier_model.predict(X_test_scaled)

# Compare the model's predictions to the real, correct answers.
accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, average="weighted")

print("Accuracy:", round(accuracy * 100, 2), "%")
print("F1 Score:", round(f1, 4))
print("")
print("Detailed report:")
print(classification_report(y_test, predictions))

In [ ]:
# BONUS: Let's see which features the model thinks are the MOST important.
# This is what makes our model "explainable" instead of a black box.

importance_values = classifier_model.feature_importances_

feature_importance_table = pd.DataFrame({
    "feature": feature_columns,
    "importance": importance_values
})

feature_importance_table = feature_importance_table.sort_values(
    by="importance",
    ascending=False
)

print(feature_importance_table)

## SECTION 10: Train the Anomaly Detector

This second model has a different job. Instead of predicting a category, it looks at every signal and asks:

**"Does this look statistically normal, or does it look weird/unusual compared to everything else?"**

We use an **Isolation Forest**. In simple words: unusual data points are easier to "isolate" (separate from the rest) than normal ones, because normal points are surrounded by lots of similar points. This model measures how easy each point is to isolate.

In [ ]:
# contamination = our rough guess of what percentage of signals are likely to be unusual.
# We are guessing 5% here, which is a common beginner-safe starting point.

anomaly_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)

# The anomaly detector learns from ALL the scaled data (not split into train/test),
# because its job is to describe the data itself, not predict on new data.
all_data_scaled = scaler.transform(X)

anomaly_model.fit(all_data_scaled)

print("Anomaly detector training complete.")

In [ ]:
# Let's check how many signals got flagged as unusual.

anomaly_predictions = anomaly_model.predict(all_data_scaled)
# Isolation Forest gives -1 for "unusual" and 1 for "normal"

number_of_anomalies = (anomaly_predictions == -1).sum()
number_of_normal = (anomaly_predictions == 1).sum()

print("Normal signals:", number_of_normal)
print("Unusual signals flagged:", number_of_anomalies)

## SECTION 11: Combine Everything — Priority Score and Fun Label

Now we combine both models into something the frontend can actually use:

1. **Priority Score** — a simple, transparent number (0 to 100) telling us how worth-investigating a signal is. Higher = more worth a human's attention.
2. **Fun Label** — a playful text label for signals flagged as unusual. This is just a presentation layer on top of REAL statistics — it is not making anything up, just describing a genuine anomaly in a more exciting way.

In [ ]:
def calculate_priority_score(classifier_confidence, is_anomaly):
    """
    This function turns model outputs into one simple 0-100 score.

    classifier_confidence: a number between 0 and 1, how confident
                            the classifier is that this is a real planet
    is_anomaly: True or False, whether the anomaly detector flagged this signal
    """

    # Start with the classifier's confidence, turned into a 0-100 scale.
    base_score = classifier_confidence * 100

    # If the signal is also flagged as unusual, we boost the score a little.
    # Why? Because unusual + possibly-real signals are exactly the ones
    # scientists want to look at more closely.
    if is_anomaly:
        base_score = base_score + 10

    # Make sure the score never goes above 100.
    if base_score > 100:
        base_score = 100

    return round(base_score, 2)


def get_fun_label(is_anomaly, anomaly_score):
    """
    Turns a real anomaly detection result into a fun, presentable label.
    This does NOT invent fake science. It just describes a real
    statistical outlier in a more exciting, human-readable way.
    """

    if is_anomaly == False:
        return "Standard Signal"

    # The more negative the anomaly_score, the more unusual the point is.
    if anomaly_score < -0.15:
        return "Highly Unusual Signal - Priority Review"
    else:
        return "Unusual Cosmic Signature Detected"


print("Both helper functions are ready.")

In [ ]:
# Let's test both functions on a few real examples from our test data,
# just to see everything working together.

sample_rows = X_test.head(5)
sample_rows_scaled = scaler.transform(sample_rows)

# Get classifier predictions and confidence scores.
sample_predictions = classifier_model.predict(sample_rows_scaled)
sample_probabilities = classifier_model.predict_proba(sample_rows_scaled)

# Get anomaly results.
sample_anomaly_flags = anomaly_model.predict(sample_rows_scaled)
sample_anomaly_scores = anomaly_model.decision_function(sample_rows_scaled)

for i in range(len(sample_rows)):
    predicted_class = sample_predictions[i]
    confidence = sample_probabilities[i].max()
    is_anomaly = (sample_anomaly_flags[i] == -1)
    anomaly_score_value = sample_anomaly_scores[i]

    priority = calculate_priority_score(confidence, is_anomaly)
    fun_label = get_fun_label(is_anomaly, anomaly_score_value)

    print("Row", i)
    print("  Predicted class:", predicted_class)
    print("  Confidence:", round(confidence, 2))
    print("  Priority score:", priority)
    print("  Label:", fun_label)
    print("")

## SECTION 12: Save the Trained Models to Files

We save 3 files:
1. `stellarscan_classifier.joblib` -> the Random Forest classification model
2. `stellarscan_anomaly_detector.joblib` -> the Isolation Forest anomaly model
3. `stellarscan_scaler.joblib` -> the scaler (needed later, so new data gets scaled the exact same way)

All three are needed by the Flask backend later, so do not skip saving any of them.

In [ ]:
joblib.dump(classifier_model, "stellarscan_classifier.joblib")
joblib.dump(anomaly_model, "stellarscan_anomaly_detector.joblib")
joblib.dump(scaler, "stellarscan_scaler.joblib")

print("All 3 model files saved in the Colab workspace.")

## SECTION 13: Download the Model Files

Run the cell below. Colab will pop up 3 separate download prompts, one for each file. Save all 3 into your project's `backend/model/` folder.

In [ ]:
from google.colab import files

files.download("stellarscan_classifier.joblib")
files.download("stellarscan_anomaly_detector.joblib")
files.download("stellarscan_scaler.joblib")

## ✅ Done!

You now have 3 trained model files. Next step (separate from this notebook): a Flask backend that loads these 3 files and exposes a `/predict` endpoint for your frontend to call.

Keep note of the exact order of `feature_columns` from Section 4 — the Flask backend must send features to the model in this exact same order:

`koi_period, koi_duration, koi_depth, koi_prad, koi_teq, koi_insol, koi_model_snr, koi_steff, koi_slogg, koi_srad`